# Paper 2 — BERT for Fake News Detection

**Reference:** J. Devlin, M.-W. Chang, K. Lee, and K. Toutanova, *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*, NAACL-HLT, 2019.

**Pipeline:** labeled news → BERT tokenizer → BERT sequence classifier → fine-tuning → evaluation.

> The notebook calculates the result from your dataset; it does not hard-code the main project's 89% accuracy.


In [ ]:
# Install if needed:
# !pip install pandas numpy scikit-learn matplotlib torch transformers datasets accelerate

import os, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay, classification_report
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding, set_seed

SEED=42
set_seed(SEED)
MODEL_NAME="bert-base-uncased"
DATA_PATH="fake_news_dataset.csv"  # <-- change this to your CSV


In [ ]:
df=pd.read_csv(DATA_PATH)
print("Shape:",df.shape)
print("Columns:",list(df.columns))
display(df.head())


In [ ]:
text_candidates=["text","content","article","news","body","title"]
label_candidates=["label","class","target","category"]
lm={c.lower():c for c in df.columns}
text_col=next((lm[x] for x in text_candidates if x in lm),None)
label_col=next((lm[x] for x in label_candidates if x in lm),None)
if text_col is None or label_col is None:
    raise ValueError("Set text_col and label_col manually after inspecting df.columns.")
print("Text:",text_col," Label:",label_col)


In [ ]:
work=df[[text_col,label_col]].copy()
work.columns=["text","label"]
work=work.dropna()
def encode_labels(s):
    s=s.astype(str).str.strip().str.lower()
    mapping={"fake":0,"false":0,"0":0,"real":1,"true":1,"1":1}
    if set(s.unique()).issubset(mapping): return s.map(mapping).astype(int)
    u=sorted(s.unique())
    if len(u)==2:
        m={u[0]:0,u[1]:1}; print("Automatic label mapping:",m); return s.map(m).astype(int)
    raise ValueError("Binary labels required.")
work["label"]=encode_labels(work["label"])
work["text"]=work["text"].astype(str)


In [ ]:
train_df,test_df=train_test_split(work,test_size=0.20,random_state=SEED,stratify=work["label"])
train_ds=Dataset.from_pandas(train_df[["text","label"]].reset_index(drop=True))
test_ds=Dataset.from_pandas(test_df[["text","label"]].reset_index(drop=True))
print("Train:",len(train_ds)," Test:",len(test_ds))


## BERT tokenization

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize(batch):
    return tokenizer(batch["text"],truncation=True,max_length=128)
train_tok=train_ds.map(tokenize,batched=True)
test_tok=test_ds.map(tokenize,batched=True)
collator=DataCollatorWithPadding(tokenizer=tokenizer)


In [ ]:
model=AutoModelForSequenceClassification.from_pretrained(MODEL_NAME,num_labels=2)

def compute_metrics(eval_pred):
    logits,labels=eval_pred
    p=np.argmax(logits,axis=-1)
    precision,recall,f1,_=precision_recall_fscore_support(labels,p,average="binary",zero_division=0)
    return {"accuracy":accuracy_score(labels,p),"precision":precision,"recall":recall,"f1":f1}


## Fine-tune BERT

Two epochs with a small batch size are used as a practical lab configuration. Training can take substantial time without a GPU.


In [ ]:
args=TrainingArguments(
    output_dir="./bert_results",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    seed=SEED,
    fp16=torch.cuda.is_available()
)
trainer=Trainer(
    model=model,args=args,train_dataset=train_tok,eval_dataset=test_tok,
    tokenizer=tokenizer,data_collator=collator,compute_metrics=compute_metrics
)


In [ ]:
trainer.train()


## Evaluation

In [ ]:
ev=trainer.evaluate()
bert_accuracy=ev["eval_accuracy"]
print(ev)
print(f"BERT accuracy: {bert_accuracy*100:.2f}%")


In [ ]:
out=trainer.predict(test_tok)
y_true=np.array(test_tok["label"])
y_pred=np.argmax(out.predictions,axis=-1)
print(classification_report(y_true,y_pred,target_names=["Fake","Real"],zero_division=0))

cm=confusion_matrix(y_true,y_pred)
fig,ax=plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay(cm,display_labels=["Fake","Real"]).plot(ax=ax)
ax.set_title("Paper 2 — BERT")
plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(5,4))
plt.bar(["BERT"],[bert_accuracy*100])
plt.ylabel("Accuracy (%)"); plt.ylim(0,100)
plt.title("Paper 2 Accuracy")
plt.tight_layout(); plt.show()


## Example prediction

In [ ]:
def predict_news(text):
    inputs=tokenizer(text,return_tensors="pt",truncation=True,max_length=128)
    inputs={k:v.to(model.device) for k,v in inputs.items()}
    with torch.no_grad():
        probs=torch.softmax(model(**inputs).logits,dim=-1)[0]
    p=int(torch.argmax(probs))
    return ("REAL" if p==1 else "FAKE"),float(probs[p])

for text in [
    "The Eiffel Tower is located in Paris.",
    "The Eiffel Tower is located in London."
]:
    label,conf=predict_news(text)
    print(f"{text}\nPrediction: {label} | Confidence: {conf:.3f}\n")


## Comparison with the reference paper

Enter the accuracy reported by the specific paper you are comparing against. Do not assume that the paper's result will be reproduced by this experimental setup.


In [ ]:
REFERENCE_PAPER_ACCURACY=None  # e.g. 92.0
if REFERENCE_PAPER_ACCURACY is None:
    print(f"Our BERT accuracy: {bert_accuracy*100:.2f}%")
    print("Set REFERENCE_PAPER_ACCURACY to compare.")
else:
    print(f"Our BERT accuracy: {bert_accuracy*100:.2f}%")
    print(f"Reference: {REFERENCE_PAPER_ACCURACY:.2f}%")
    print(f"Difference: {bert_accuracy*100-REFERENCE_PAPER_ACCURACY:+.2f} percentage points")


## Save the fine-tuned model

The saved model can be reused by the main project. Model weights may be too large for ordinary GitHub uploads, so do not commit them unless you intentionally use Git LFS or another model host.


In [ ]:
SAVE_DIR="./saved_bert_model"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved to",SAVE_DIR)
